## Demo

In [ ]:
import os
import sys
import platform

from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
chrome_options.add_experimental_option('useAutomationExtension', False)


path_to_exe = "chromedriver-win64/chromedriver.exe"
service = Service(path_to_exe)
driver = webdriver.Chrome(service=service, 
                          #options=chrome_options
                         )

司法院判決書系統

抓取所有的判決書url和metadata，像是時間，案件類別等等

從 url https://judgment.judicial.gov.tw/FJUD/Default_AD.aspx 讀取網頁內容

In [ ]:
def access_court_verdict_urls(driver, url: str):

    main_tab = driver.current_window_handle
    
    driver.switch_to.new_window('tab')

    all_urls = None
    
    while True:
        driver.get(url)
        
        jud =  WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, 'jud'))
        )    
    
        all_verdicts = jud.find_elements(By.TAG_NAME, 'a')
    
        if not all_urls:
            all_urls = [verdict.get_attribute("href") for verdict in all_verdicts]
        else:
            all_urls += [verdict.get_attribute("href") for verdict in all_verdicts]

        try:
            next_page_icon = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, 'hlNext'))
            )

            url = next_page_icon.get_attribute("href")
            
        except:
            break

    driver.close()

    driver.switch_to.window(main_tab)
    
    return all_urls

In [ ]:
from typing import List, Dict

type_name_dict = {"C": "憲法",
                  "V": "民事",
                  "M": "刑事",
                  "A": "行政",
                  "P": "懲戒"}

def fill_placeholder(driver, content, id: str):

    placeholder = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, id))
        ) 

    placeholder.send_keys(content)


def retrieve_case_by_day(year, month, day) -> List[Dict]:

    main_url = "https://judgment.judicial.gov.tw/FJUD/Default_AD.aspx"

    driver = webdriver.Chrome(service=service, 
                          #options=chrome_options
                         )

    driver.set_page_load_timeout(15)
    
    driver.get(main_url)

    output = []
    
    for idx, type_name in enumerate(["C", "V", "M", "A", "P"]):
        
        to_be_click = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, f'input[type="checkbox"][name="jud_sys"][value={type_name}]')))
        
        to_be_click.click()

        # unclick checkbox
        for prev_type_name in ["C", "V", "M", "A", "P"][:idx]:
            to_be_unclicked = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, f'input[type="checkbox"][name="jud_sys"][value={prev_type_name}]')))
            if to_be_unclicked.is_selected():        
                to_be_unclicked.click()

        fill_placeholder(driver, content=year, id='dy1')
        fill_placeholder(driver, content=month, id='dm1')
        fill_placeholder(driver, content=day, id='dd1')
        fill_placeholder(driver, content=year, id='dy2')
        fill_placeholder(driver, content=month, id='dm2')
        fill_placeholder(driver, content=day, id='dd2')
    
        submit = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "btnQry")))

        submit.click()
        
        tabcontent = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div[class='tab-content']"))
        )
        if tabcontent.text == "查詢結果\n0":
            print(f"{type_name_dict[type_name]} on {year}-{month}-{day} has not result")
            driver.get(main_url)
            continue
        else:
            dvGrpCourt = WebDriverWait(tabcontent, 10).until(
                EC.presence_of_element_located((By.ID, "dvGrpCourt"))
            )

        all_court_elements = dvGrpCourt.find_elements(By.TAG_NAME, "li")

        for court_element in all_court_elements:

            main_tab = driver.current_window_handle
            
            link = WebDriverWait(court_element, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "a")))

            court_url = link.get_attribute("href")
            court_name = link.text.split("\n")[0]

            court_verdict_urls = access_court_verdict_urls(driver, url=court_url)

            print(f"{type_name_dict[type_name]} @ {court_name} on {year}-{month}-{day} has {len(court_verdict_urls)} results")
            
            driver.switch_to.window(main_tab)

            metadata = [{"url": court_verdict_url,
                         "type": type_name_dict[type_name],
                         "court": court_name,
                         "year": year,
                         "month": month,
                         "day": day} for court_verdict_url in court_verdict_urls]

            output.extend(metadata)
        driver.get(main_url)
        driver.delete_all_cookies()

    return output

選取時間:

- 民國年 114
- 月: 11
- 日期: 20

In [ ]:
exp_urls = retrieve_case_by_day(114, 11, 20)